# Anomaly Detection: A Comprehensive Guide

### From Statistical Foundations to Modern Machine Learning Approaches

---

| Chapter | Topic | Key Algorithms |
| --- | --- | --- |
| 1 | Introduction to Anomaly Detection | Definitions, motivation, formal framework |
| 2 | Types of Anomalies and Taxonomy of Methods | Point, contextual, collective anomalies |
| 3 | Statistical Approaches | Z-Score, IQR, Grubbs' Test |
| 4 | Isolation Forest | iTree construction, anomaly score derivation |
| 5 | Local Outlier Factor (LOF) | k-distance, reachability density, LOF score |
| 6 | One-Class SVM | Primal/dual optimization, kernel trick |
| 7 | DBSCAN for Anomaly Detection | Core, border, and noise points |
| 8 | Elliptic Envelope | Mahalanobis distance, robust covariance |
| 9 | Autoencoders for Anomaly Detection | Reconstruction error, encoder-decoder architecture |
| 10 | Comparative Analysis | Strengths, weaknesses, decision framework |

## Chapter 1: Introduction to Anomaly Detection

**Anomaly detection** (also called *outlier detection*) is the task of identifying observations that deviate significantly from the expected behavior of a dataset. These rare items, events, or observations raise suspicion because they differ substantially from the majority of the data.

### 1.1 Why Does Anomaly Detection Matter?

Anomalies, though rare, often carry the most critical information in a dataset:

| Domain | Normal Behavior | Anomaly | Business Impact |
| --- | --- | --- | --- |
| Banking | Routine transactions | Fraudulent charges | Financial loss prevention |
| Manufacturing | Stable sensor readings | Equipment malfunction | Predictive maintenance |
| Cybersecurity | Standard network traffic | Intrusion attempts | Threat detection |
| Healthcare | Normal vital signs | Irregular heartbeat | Early diagnosis |
| E-commerce | Organic user behavior | Bot activity, click fraud | Revenue protection |

### 1.2 Formal Definition

Given a dataset $$\mathcal{D} = \{x_1, x_2, \ldots, x_n\}$$ where each $$x_i \in \mathbb{R}^d$$, the goal of anomaly detection is to find a **scoring function** $$s: \mathbb{R}^d \rightarrow \mathbb{R}$$ such that:

$$
s(x_i) > \tau \implies x_i \text{ is classified as an anomaly}
$$

where $$\tau$$ is a decision threshold. Equivalently, we seek a **decision function** $$f: \mathbb{R}^d \rightarrow \{-1, +1\}$$ where $$f(x) = -1$$ indicates an anomaly and $$f(x) = +1$$ indicates a normal instance.

### 1.3 Key Challenges

* **Class Imbalance**: Anomalies are typically $$< 1\%$$ of the data, making supervised classification heavily biased
* **Lack of Labels**: In most real-world settings, we do not have labeled anomalies for training
* **Novel Anomalies**: The system must detect previously unseen *types* of anomalies (zero-day attacks, new fraud patterns)
* **High Dimensionality**: The curse of dimensionality makes distance and density estimation less reliable in $$\mathbb{R}^d$$ for large $$d$$
* **Noise vs. Anomaly**: Distinguishing genuine anomalies from noisy but benign observations requires careful calibration
* **Concept Drift**: In streaming settings, the definition of "normal" can evolve over time

## Chapter 2: Types of Anomalies and Taxonomy of Methods

### 2.1 Types of Anomalies

**Point Anomalies** 
A single data instance is anomalous with respect to the rest of the data. This is the simplest type of anomaly.

*Example*: A credit card transaction of $$\$50{,}000$$ when all historical transactions are below $$\$500$$.

Formally, a point $$x_i$$ is a point anomaly if:

$$
P(x_i \mid \mathcal{D} \setminus \{x_i\}) < \epsilon
$$

for some small threshold $$\epsilon$$.

**Contextual (Conditional) Anomalies** 
A data instance is anomalous *in a specific context* but not otherwise. The data has two types of attributes:
* *Contextual attributes*: define the context (e.g., time of day, geographic location)
* *Behavioral attributes*: define the non-contextual characteristics (e.g., transaction amount, temperature)

*Example*: A temperature of $$35°C$$ is normal in July but anomalous in January for a city in the Northern Hemisphere.

**Collective Anomalies** 
A *collection* of related data instances is anomalous, even though individual instances may not be anomalous by themselves.

*Example*: A sequence of 50 small transactions ($$\$1$$ each) within 2 minutes. Each $$\$1$$ transaction is normal individually, but the rapid sequence indicates credit card "testing" by fraudsters.

### 2.2 Taxonomy of Detection Methods

| Category | Methods | Core Principle | Strengths | Limitations |
| --- | --- | --- | --- | --- |
| **Statistical** | Z-Score, IQR, Grubbs' | Data follows a known distribution | Simple, interpretable | Parametric assumptions |
| **Density-based** | LOF, LOCI, COF | Anomalies reside in low-density regions | Captures local structure | Expensive for large $$n$$ |
| **Distance-based** | KNN-based, ODIN | Anomalies are far from their neighbors | Intuitive | Sensitive to $$k$$ and metric |
| **Isolation-based** | Isolation Forest, SCiForest | Anomalies are easier to isolate | Fast, scalable, no density estimation | Struggles with local anomalies |
| **Boundary-based** | One-Class SVM, SVDD | Learn a tight boundary around normal data | Powerful with kernels | Kernel and $$\nu$$ tuning |
| **Clustering-based** | DBSCAN, k-Means variant | Anomalies do not belong to any cluster | Discovers structure | Assumes data is clusterable |
| **Reconstruction** | Autoencoders, PCA | Anomalies have high reconstruction error | Handles complex manifolds | Requires training; opaque |

### 2.3 Learning Paradigms

* **Supervised**: Requires labeled examples of both normal and anomalous classes. Rarely feasible because anomalies are scarce and novel types may not appear in training data. Reduces to a heavily imbalanced classification problem.
* **Semi-supervised**: Trained on clean (normal-only) data. Learns a model of normality; anything deviating from it is flagged. One-Class SVM and Autoencoders naturally fit this paradigm.
* **Unsupervised**: No labels required at all. Algorithms like Isolation Forest, LOF, and DBSCAN operate purely on the structure of the data. *This is the most common and practical setting.*

## Chapter 3: Statistical Approaches to Anomaly Detection

Statistical methods are the oldest and most interpretable family of anomaly detectors. They assume the data is generated from a known (or estimable) distribution and flag points that have low probability under that distribution.

### 3.1 Z-Score Method

**Intuition**: If data follows a Gaussian distribution, points far from the mean (in units of standard deviation) are unlikely and therefore anomalous.

For a univariate dataset, the Z-score of observation $$x_i$$ is:

$$
z_i = \frac{x_i - \mu}{\sigma}
$$

where $$\mu = \frac{1}{n}\sum_{i=1}^n x_i$$ is the sample mean and $$\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^n (x_i - \mu)^2}$$ is the sample standard deviation.

**Decision rule**: Flag $$x_i$$ as anomalous if $$|z_i| > \tau$$, where $$\tau$$ is typically 2.5 or 3.

Under a true Gaussian, $$P(|Z| > 3) \approx 0.0027$$, so roughly 0.27% of data would be flagged.

**Multivariate extension**: For $$x \in \mathbb{R}^d$$, we use the Mahalanobis distance (covered in Chapter 8).

**Limitation**: Highly sensitive to the mean and variance estimates, which are themselves influenced by outliers. This is a *masking effect* where extreme outliers inflate $$\sigma$$, making other outliers appear less extreme.

### 3.2 Interquartile Range (IQR) Method

**Intuition**: Use robust statistics (quartiles) instead of mean/variance to avoid the masking effect.

Let $$Q_1$$ and $$Q_3$$ denote the 25th and 75th percentiles, and $$\text{IQR} = Q_3 - Q_1$$.

**Decision rule**: Flag $$x_i$$ as anomalous if:

$$
x_i < Q_1 - k \cdot \text{IQR} \quad \text{or} \quad x_i > Q_3 + k \cdot \text{IQR}
$$

where $$k = 1.5$$ flags "mild" outliers and $$k = 3.0$$ flags "extreme" outliers.

**Why 1.5?** For a Gaussian distribution, the range $$[Q_1 - 1.5 \cdot \text{IQR},\; Q_3 + 1.5 \cdot \text{IQR}]$$ contains approximately 99.3% of the data, making it a reasonable default.

### 3.3 Grubbs' Test

**Intuition**: A formal hypothesis test for a single outlier in a univariate Gaussian sample.

**Null hypothesis** $$H_0$$: There are no outliers in the dataset. 
**Alternative** $$H_1$$: There is exactly one outlier.

The Grubbs test statistic is:

$$
G = \frac{\max_{i} |x_i - \bar{x}|}{s}
$$

where $$\bar{x}$$ is the sample mean and $$s$$ is the sample standard deviation. Reject $$H_0$$ at significance level $$\alpha$$ if:

$$
G > \frac{n - 1}{\sqrt{n}} \sqrt{\frac{t^2_{\alpha/(2n),\, n-2}}{n - 2 + t^2_{\alpha/(2n),\, n-2}}}
$$

where $$t_{\alpha/(2n),\, n-2}$$ is the critical value of the t-distribution with $$n-2$$ degrees of freedom.

### 3.4 Industrial Example: Quality Control in Beverage Manufacturing

A bottling plant fills containers to a target volume of 500 mL. Sensors measure the actual fill volume for each bottle. Statistical methods can flag bottles that are significantly under-filled (loss for the customer) or over-filled (loss for the manufacturer). The Z-score method with $$\tau = 3$$ would flag any bottle more than 3 standard deviations from the mean fill volume.

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# --- Simulate beverage fill volumes (mL) ---
normal_volumes = np.random.normal(loc=500, scale=2, size=500)       # normal bottles
anomalous_volumes = np.array([488, 490, 515, 518, 485])              # defective bottles
volumes = np.concatenate([normal_volumes, anomalous_volumes])
np.random.shuffle(volumes)

# =============================================================
# METHOD 1: Z-Score
# =============================================================
mean_vol = np.mean(volumes)
std_vol  = np.std(volumes)
z_scores = (volumes - mean_vol) / std_vol

z_threshold = 3
z_anomalies = np.where(np.abs(z_scores) > z_threshold)[0]

print("=== Z-Score Method ===")
print(f"Mean: {mean_vol:.2f} mL, Std: {std_vol:.2f} mL")
print(f"Anomalies detected: {len(z_anomalies)}")
print(f"Anomalous volumes: {volumes[z_anomalies]}")

# =============================================================
# METHOD 2: IQR
# =============================================================
Q1 = np.percentile(volumes, 25)
Q3 = np.percentile(volumes, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

iqr_anomalies = np.where((volumes < lower_bound) | (volumes > upper_bound))[0]

print("\n=== IQR Method ===")
print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Anomalies detected: {len(iqr_anomalies)}")
print(f"Anomalous volumes: {volumes[iqr_anomalies]}")

# =============================================================
# METHOD 3: Grubbs' Test (iterative)
# =============================================================
def grubbs_test(data, alpha=0.05):
    """Perform one iteration of Grubbs' test."""
    n = len(data)
    mean_val = np.mean(data)
    std_val  = np.std(data, ddof=1)
    G = np.max(np.abs(data - mean_val)) / std_val
    # Critical value
    t_crit = stats.t.ppf(1 - alpha / (2 * n), n - 2)
    G_crit = ((n - 1) / np.sqrt(n)) * np.sqrt(t_crit**2 / (n - 2 + t_crit**2))
    return G, G_crit, G > G_crit

G_stat, G_crit, is_outlier = grubbs_test(volumes)
print("\n=== Grubbs' Test ===")
print(f"G statistic: {G_stat:.4f}, Critical value: {G_crit:.4f}")
print(f"Outlier detected: {is_outlier}")

# =============================================================
# VISUALIZATION
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Z-Score plot
axes[0].scatter(range(len(volumes)), volumes, c='steelblue', alpha=0.5, s=10)
axes[0].scatter(z_anomalies, volumes[z_anomalies], c='red', s=50, label='Anomaly', zorder=5)
axes[0].axhline(mean_vol + z_threshold * std_vol, color='red', linestyle='--', alpha=0.7)
axes[0].axhline(mean_vol - z_threshold * std_vol, color='red', linestyle='--', alpha=0.7)
axes[0].set_title('Z-Score Method', fontsize=13)
axes[0].set_ylabel('Volume (mL)')
axes[0].legend()

# IQR plot
axes[1].boxplot(volumes, vert=True)
axes[1].set_title('IQR Method (Box Plot)', fontsize=13)
axes[1].set_ylabel('Volume (mL)')

# Distribution with Grubbs
axes[2].hist(volumes, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
axes[2].axvline(mean_vol + G_crit * std_vol, color='red', linestyle='--', label=f'Grubbs critical')
axes[2].axvline(mean_vol - G_crit * std_vol, color='red', linestyle='--')
axes[2].set_title("Grubbs' Test", fontsize=13)
axes[2].set_xlabel('Volume (mL)')
axes[2].legend()

plt.suptitle('Statistical Anomaly Detection — Beverage Fill Volume QC', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## Chapter 4: Isolation Forest

> *"Anomalies are few and different — and therefore easier to isolate."* 
> — Liu, Ting, and Zhou (2008)

### 4.1 Core Intuition

Isolation Forest (iForest) takes a fundamentally different approach from density or distance-based methods. Instead of *profiling normal instances*, it directly **isolates anomalies**.

The key insight rests on two properties of anomalies:
* **Fewer in number**: anomalies constitute a small fraction of the data
* **Different in attribute values**: anomalies have feature values that are far from those of normal instances

Because of these two properties, anomalies are *susceptible to isolation* — they can be separated from the rest of the data with very few random partitions.

**Thought experiment**: Imagine you have 1000 data points in 2D, with 995 clustered tightly and 5 scattered far away. If you randomly pick a feature and a random split value, the 5 scattered points will be isolated in just 1–2 splits, while a point deep inside the cluster will require many splits to be separated from its neighbors.

### 4.2 The Isolation Tree (iTree)

An **Isolation Tree** is a binary tree built by the following recursive procedure:

---
**Algorithm: iTree Construction**

**Input**: Dataset $$X = \{x_1, \ldots, x_n\}$$, current height $$e$$, height limit $$l$$

1. If $$|X| \leq 1$$ or $$e \geq l$$: return a **leaf node** (external node) with size $$|X|$$
2. Randomly select a feature $$q$$ from the $$d$$ available features
3. Randomly select a split value $$p$$ uniformly from $$[\min(X_q),\; \max(X_q)]$$
4. Partition $$X$$ into:
   * $$X_L = \{x \in X \mid x_q < p\}$$
   * $$X_R = \{x \in X \mid x_q \geq p\}$$
5. Return an **internal node** with left child $$\text{iTree}(X_L, e+1, l)$$ and right child $$\text{iTree}(X_R, e+1, l)$$

---

The **height limit** is set to $$l = \lceil \log_2 \psi \rceil$$ where $$\psi$$ is the subsampling size. This is because the average tree height grows logarithmically, and we only care about shorter-than-average paths (which indicate anomalies).

### 4.3 The Isolation Forest (Ensemble)

---
**Algorithm: Isolation Forest**

**Input**: Dataset $$X$$, number of trees $$t$$, subsample size $$\psi$$

1. **For** $$i = 1, \ldots, t$$:
   * Draw a random subsample $$X_i \subset X$$ of size $$\psi$$ (without replacement)
   * Build $$\text{iTree}_i$$ from $$X_i$$ with height limit $$l = \lceil \log_2 \psi \rceil$$
2. **For** each test instance $$x$$:
   * Traverse each tree and record the **path length** $$h_i(x)$$
   * Compute the average path length $$E[h(x)] = \frac{1}{t}\sum_{i=1}^t h_i(x)$$
   * Compute the **anomaly score** $$s(x, \psi)$$

---

**Path length** $$h(x)$$ is the number of edges $$x$$ traverses from the root to the terminating external node. When a data point reaches a leaf node before complete isolation (because the height limit was reached or the leaf contains multiple points), an adjustment term is added:

$$
h(x) = e + c(\text{size})
$$

where $$e$$ is the actual number of edges traversed, and $$c(\text{size})$$ is the average path length of an unsuccessful search in a BST with $$\text{size}$$ items (explained in the next section). This corrects for the early termination.

### 4.4 Why Subsampling?

Subsampling (typically $$\psi = 256$$) is not just a computational shortcut — it actually *improves* detection quality:

* **Reduces swamping**: With fewer points, anomalies are less likely to be "hidden" by nearby normal points
* **Reduces masking**: Normal clusters become less dense, making anomalies relatively easier to isolate
* **Computational efficiency**: Each tree is built in $$O(\psi \log \psi)$$ time instead of $$O(n \log n)$$

### 4.5 Hyperparameters

| Parameter | Typical Value | Effect |
| --- | --- | --- |
| $$t$$ (number of trees) | 100–300 | More trees reduce variance; 100 usually suffices |
| $$\psi$$ (subsample size) | 256 | Larger values may mask anomalies; smaller values increase variance |
| contamination | 0.01–0.1 | Sets the anomaly score threshold (expected fraction of outliers) |

### 4.6 The Anomaly Score — Detailed Derivation

The anomaly score normalizes the average path length against the expected path length in a comparable random structure. This section provides the complete derivation.

#### Step 1: Connection to Binary Search Trees (BSTs)

An isolation tree that randomly partitions data is structurally equivalent to a **Binary Search Tree (BST)** where keys arrive in random order. In both cases:
* Each internal node splits data into two branches
* The position of a data point is determined by a sequence of random comparisons

Therefore, the expected path length of a data point in an iTree follows the same distribution as the **unsuccessful search path length** in a BST.

#### Step 2: Deriving $$c(n)$$ — Average Path Length in a BST

Let $$C(n)$$ denote the expected number of edges traversed in an **unsuccessful search** in a BST built from $$n$$ randomly ordered keys. A BST with $$n$$ internal nodes has $$n + 1$$ external (null) nodes.

Let $$E_n$$ be the **total external path length** (sum of depths of all external nodes). Then:

$$
c(n) = \frac{E_n}{n + 1}
$$

To derive $$E_n$$, consider that when the root is chosen (equally likely to be any of the $$n$$ keys), it divides the remaining $$n-1$$ keys into a left subtree of size $$k$$ and a right subtree of size $$n - 1 - k$$, for $$k = 0, 1, \ldots, n-1$$, each with probability $$\frac{1}{n}$$.

Every external node's path passes through the root, contributing +1 to its depth. Since there are $$n + 1$$ external nodes:

$$
E_n = (n + 1) + \frac{1}{n} \sum_{k=0}^{n-1} \big[ E_k + E_{n-1-k} \big]
$$

By symmetry, $$\sum_{k=0}^{n-1} E_k = \sum_{k=0}^{n-1} E_{n-1-k}$$, so:

$$
E_n = (n + 1) + \frac{2}{n} \sum_{k=0}^{n-1} E_k
$$

with base case $$E_0 = 0$$ (an empty tree has zero path length).

#### Step 3: Solving the Recurrence

Multiply both sides by $$n$$:

$$
n \cdot E_n = n(n+1) + 2\sum_{k=0}^{n-1} E_k \tag{A}
$$

Write the same for $$n-1$$:

$$
(n-1) \cdot E_{n-1} = (n-1)n + 2\sum_{k=0}^{n-2} E_k \tag{B}
$$

Subtract (B) from (A):

$$
n \cdot E_n - (n-1) \cdot E_{n-1} = n(n+1) - (n-1)n + 2E_{n-1}
$$

$$
n \cdot E_n - (n-1) \cdot E_{n-1} = 2n + 2E_{n-1}
$$

$$
n \cdot E_n = (n+1) \cdot E_{n-1} + 2n
$$

Divide both sides by $$n(n+1)$$:

$$
\frac{E_n}{n+1} = \frac{E_{n-1}}{n} + \frac{2}{n+1}
$$

Let $$a_n = \frac{E_n}{n+1} = c(n)$$. Then:

$$
a_n = a_{n-1} + \frac{2}{n+1}
$$

Telescopically expanding from $$a_1 = c(1) = 0$$:

$$
a_n = \sum_{k=2}^{n+1} \frac{2}{k} = 2\sum_{k=2}^{n+1} \frac{1}{k} = 2\left(\sum_{k=1}^{n+1} \frac{1}{k} - 1\right) = 2\left(H_{n+1} - 1\right) = 2H_n - \frac{2n}{n+1}
$$

Wait — let us be more precise. Using the Isolation Forest paper's convention where the subsample has $$\psi$$ points:

$$
\boxed{c(n) = 2H(n-1) - \frac{2(n-1)}{n}}
$$

where the harmonic number is approximated by:

$$
H(i) = \ln(i) + \gamma \approx \ln(i) + 0.5772156649
$$

and $$\gamma$$ is the **Euler–Mascheroni constant**.

**Boundary cases**:

$$
c(n) = \begin{cases} 2H(n-1) - \frac{2(n-1)}{n} & \text{if } n > 2 \\ 1 & \text{if } n = 2 \\ 0 & \text{if } n \leq 1 \end{cases}
$$

#### Step 4: The Anomaly Score

Given the average path length $$E[h(x)]$$ of instance $$x$$ across all $$t$$ trees in the forest, the **anomaly score** is:

$$
\boxed{s(x, \psi) = 2^{\displaystyle -\frac{E[h(x)]}{c(\psi)}}}
$$

where $$c(\psi)$$ is the normalization constant derived above, and $$\psi$$ is the subsample size.

#### Step 5: Score Interpretation

* If $$E[h(x)] \to 0$$: the instance is isolated almost immediately. $$s \to 2^{0} = 1$$. **Definite anomaly.**
* If $$E[h(x)] = c(\psi)$$: the instance has an average path length equal to the expected depth. $$s = 2^{-1} = 0.5$$. **No clear signal — could be either.**
* If $$E[h(x)] \to n - 1$$: the instance requires nearly the maximum number of splits. $$s \to 2^{-(n-1)/c(\psi)} \approx 0$$. **Definite normal point.**

The score is bounded in $$(0, 1]$$, making it easy to interpret and threshold.

### 4.7 Time and Space Complexity

| Phase | Complexity |
| --- | --- |
| Training (building $$t$$ trees) | $$O(t \cdot \psi \log \psi)$$ |
| Scoring (per instance) | $$O(t \cdot \log \psi)$$ |
| Space (storing the forest) | $$O(t \cdot \psi)$$ |

This makes Isolation Forest one of the most computationally efficient anomaly detectors, with **linear time complexity** in the number of instances (since $$\psi$$ is a constant, typically 256).

### 4.8 Strengths and Limitations

**Strengths**:
* No distance or density computation required
* Linear time complexity; scales to millions of instances
* Effective in high dimensions (random feature selection acts as implicit feature bagging)
* Few hyperparameters

**Limitations**:
* Struggles with *local anomalies* in datasets with clusters of very different densities
* Axis-aligned splits may miss anomalies detectable only through attribute combinations
* Subsampling can miss rare but important local patterns

*Extensions like SCiForest (splitting on hyperplanes) and Extended Isolation Forest (non-axis-aligned splits) address some of these limitations.*

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

np.random.seed(42)

# --- Generate synthetic 2D data ---
cluster1 = np.random.randn(300, 2) * 0.5 + np.array([2, 2])
cluster2 = np.random.randn(300, 2) * 0.8 + np.array([-2, -2])
anomalies = np.random.uniform(low=-6, high=6, size=(20, 2))

X = np.vstack([cluster1, cluster2, anomalies])
labels_true = np.array([1]*600 + [-1]*20)

# --- Fit Isolation Forest ---
iforest = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination=0.05,
    random_state=42
)
iforest.fit(X)

y_pred = iforest.predict(X)
scores = iforest.decision_function(X)

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-7, 7, 200), np.linspace(-7, 7, 200))
Z = iforest.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

axes[0].contourf(xx, yy, Z, levels=np.linspace(Z.min(), Z.max(), 30), cmap='RdYlBu')
axes[0].contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)
normal_mask = y_pred == 1
axes[0].scatter(X[normal_mask, 0], X[normal_mask, 1], c='steelblue', s=10, alpha=0.5, label='Normal')
axes[0].scatter(X[~normal_mask, 0], X[~normal_mask, 1], c='red', s=50, marker='x', label='Anomaly')
axes[0].set_title('Isolation Forest Decision Boundary', fontsize=13)
axes[0].legend()
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Score distribution
axes[1].hist(scores[labels_true == 1], bins=40, alpha=0.7, color='steelblue', label='Normal', density=True)
axes[1].hist(scores[labels_true == -1], bins=15, alpha=0.7, color='red', label='Anomaly', density=True)
axes[1].axvline(0, color='black', linestyle='--', label='Threshold')
axes[1].set_title('Anomaly Score Distribution', fontsize=13)
axes[1].set_xlabel('Decision Function Score')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Detected anomalies: {(y_pred == -1).sum()} / {len(X)}")
print(f"True anomalies caught: {((y_pred == -1) & (labels_true == -1)).sum()} / {(labels_true == -1).sum()}")

### 4.9 Industrial Example: Credit Card Fraud Detection

**Scenario**: A major bank processes millions of credit card transactions daily. Fraudulent transactions constitute approximately 0.1–0.5% of all transactions. The bank needs a real-time system to flag suspicious transactions for human review.

**Why Isolation Forest?**
* Fraud is rare and looks "different" — exactly the properties iForest exploits
* Scales to millions of transactions with constant memory (fixed subsample size)
* No need for labeled fraud examples (unsupervised)
* Fast scoring allows real-time deployment

**Feature engineering** for credit card fraud typically includes: transaction amount (and deviation from cardholder's average), time since last transaction, geographic distance from last transaction, merchant category mismatch, and frequency of transactions in the last hour/day.

In [0]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Simulate credit card transaction data ---
n_normal, n_fraud = 9900, 100

normal_data = pd.DataFrame({
    'amount': np.random.lognormal(mean=3.5, sigma=0.8, size=n_normal),
    'time_since_last_txn': np.random.exponential(scale=5, size=n_normal),
    'geo_distance_km': np.abs(np.random.normal(loc=5, scale=10, size=n_normal)),
    'txn_count_last_hour': np.random.poisson(lam=1.5, size=n_normal),
    'merchant_risk_score': np.random.beta(a=2, b=8, size=n_normal),
})
normal_data['is_fraud'] = 0

fraud_data = pd.DataFrame({
    'amount': np.random.lognormal(mean=6.5, sigma=1.2, size=n_fraud),
    'time_since_last_txn': np.random.exponential(scale=0.3, size=n_fraud),
    'geo_distance_km': np.abs(np.random.normal(loc=500, scale=200, size=n_fraud)),
    'txn_count_last_hour': np.random.poisson(lam=12, size=n_fraud),
    'merchant_risk_score': np.random.beta(a=8, b=2, size=n_fraud),
})
fraud_data['is_fraud'] = 1

df = pd.concat([normal_data, fraud_data], ignore_index=True).sample(frac=1, random_state=42)
features = ['amount', 'time_since_last_txn', 'geo_distance_km', 'txn_count_last_hour', 'merchant_risk_score']

# --- Apply Isolation Forest ---
iforest = IsolationForest(n_estimators=300, max_samples=256, contamination=0.02, random_state=42)
df['iforest_pred'] = iforest.fit_predict(df[features])
df['iforest_score'] = -iforest.score_samples(df[features])

# --- Evaluation ---
y_true = df['is_fraud']
y_pred = (df['iforest_pred'] == -1).astype(int)

print("=== Isolation Forest: Credit Card Fraud Detection ===")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraud']))
print(f"ROC-AUC Score: {roc_auc_score(y_true, df['iforest_score']):.4f}")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'red', 'Fraud')]:
    mask = df['is_fraud'] == label
    axes[0].hist(df.loc[mask, 'iforest_score'], bins=50, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title('Anomaly Score Distribution', fontsize=13)
axes[0].set_xlabel('Anomaly Score (higher = more suspicious)')
axes[0].legend()

normal_mask = df['is_fraud'] == 0
axes[1].scatter(df.loc[normal_mask, 'amount'], df.loc[normal_mask, 'geo_distance_km'],
                c='steelblue', alpha=0.3, s=10, label='Normal')
axes[1].scatter(df.loc[~normal_mask, 'amount'], df.loc[~normal_mask, 'geo_distance_km'],
                c='red', s=40, marker='x', label='Fraud')
axes[1].set_title('Transaction Amount vs Geographic Distance', fontsize=13)
axes[1].set_xlabel('Amount ($)')
axes[1].set_ylabel('Distance from last txn (km)')
axes[1].set_xscale('log')
axes[1].legend()
plt.tight_layout()
plt.show()

## Chapter 5: Local Outlier Factor (LOF)

> *"An outlier is an observation that deviates so much from other observations as to arouse suspicion that it was generated by a different mechanism."* 
> — Hawkins (1980)

### 5.1 Core Intuition

While Isolation Forest asks "how easy is it to isolate this point?", LOF asks a fundamentally different question: **"how dense is the neighborhood of this point compared to the neighborhoods of its neighbors?"**

LOF (Breunig et al., 2000) is a *density-based* anomaly detection method. Its key insight is that **anomaly is a local property** — a point may be perfectly normal in one region of the feature space but highly anomalous in another.

**Motivating example**: Consider a dataset with two clusters. Cluster A is very dense (points are tightly packed), while Cluster B is sparse (points are spread out). A point that lies slightly outside Cluster A might be closer to Cluster A's centroid than any point in Cluster B is to Cluster B's centroid. Yet that point is anomalous *relative to Cluster A*, while points in Cluster B are normal *relative to Cluster B*.

Global methods (like Z-score or a single distance threshold) would fail here. LOF succeeds by comparing each point's density to its *local* neighborhood.

### 5.2 Core Definitions

Let $$\mathcal{D} = \{x_1, \ldots, x_n\} \subset \mathbb{R}^d$$ and $$d(A, B)$$ denote the distance between points $$A$$ and $$B$$ (typically Euclidean).

**Definition 1: $$k$$-distance** 
The $$k$$-distance of a point $$A$$, denoted $$\text{k-dist}(A)$$, is the distance between $$A$$ and its $$k$$-th nearest neighbor. Formally:

$$
\text{k-dist}(A) = d(A, B_k)
$$

where $$B_k$$ is the $$k$$-th nearest neighbor of $$A$$, i.e., there exist at most $$k - 1$$ points $$B \in \mathcal{D} \setminus \{A\}$$ with $$d(A, B) < d(A, B_k)$$.

**Definition 2: $$k$$-distance neighborhood** 
The $$k$$-distance neighborhood of $$A$$ is the set of all points within $$\text{k-dist}(A)$$:

$$
N_k(A) = \{B \in \mathcal{D} \setminus \{A\} \mid d(A, B) \leq \text{k-dist}(A)\}
$$

Note: $$|N_k(A)| \geq k$$ because ties in distance can include more than $$k$$ points.

**Definition 3: Reachability Distance** 
The reachability distance of point $$A$$ with respect to point $$B$$ is:

$$
\text{reach-dist}_k(A, B) = \max\{\text{k-dist}(B),\; d(A, B)\}
$$

**Interpretation**: If $$A$$ is very close to $$B$$, the reachability distance is "clamped" to $$\text{k-dist}(B)$$ — this smoothing effect prevents statistical fluctuations for points deep inside a cluster from causing unstable density estimates.

**Definition 4: Local Reachability Density (LRD)** 
The local reachability density of point $$A$$ is the inverse of the average reachability distance from $$A$$ to its $$k$$-nearest neighbors:

$$
\text{lrd}_k(A) = \frac{1}{\displaystyle\frac{1}{|N_k(A)|} \sum_{B \in N_k(A)} \text{reach-dist}_k(A, B)} = \frac{|N_k(A)|}{\displaystyle\sum_{B \in N_k(A)} \text{reach-dist}_k(A, B)}
$$

**Interpretation**: Higher LRD means the point lives in a denser region; lower LRD means it lives in a sparser region.

### 5.3 The LOF Score — Full Derivation

**Definition 5: Local Outlier Factor (LOF)** 
The LOF of point $$A$$ is the average ratio of the local reachability densities of $$A$$'s neighbors to $$A$$'s own LRD:

$$
\boxed{\text{LOF}_k(A) = \frac{1}{|N_k(A)|} \sum_{B \in N_k(A)} \frac{\text{lrd}_k(B)}{\text{lrd}_k(A)}}
$$

This can be rewritten as:

$$
\text{LOF}_k(A) = \frac{\displaystyle\frac{1}{|N_k(A)|} \sum_{B \in N_k(A)} \text{lrd}_k(B)}{\text{lrd}_k(A)}
$$

which is the **ratio of the average LRD of $$A$$'s neighbors to $$A$$'s own LRD**.

### 5.4 Score Interpretation

* $$\text{LOF}_k(A) \approx 1$$: Point $$A$$ has roughly the same density as its neighbors. **Normal.**
* $$\text{LOF}_k(A) \gg 1$$: Point $$A$$ has much lower density than its neighbors. Its neighbors are in a dense region, but $$A$$ itself is isolated. **Anomaly.**
* $$\text{LOF}_k(A) < 1$$: Point $$A$$ is denser than its neighbors. This can happen for points at the core of a very tight cluster. **Very normal (inlier).**

### 5.5 Detailed Walkthrough: Why LOF Works

Consider three scenarios for a point $$A$$ with $$k = 5$$:

**Case 1: $$A$$ is deep inside a dense cluster** 
All 5 neighbors are nearby $$\Rightarrow$$ $$\text{reach-dist}_k(A, B_i) \approx \text{k-dist}(B_i)$$ (clamped) $$\Rightarrow$$ $$\text{lrd}_k(A)$$ is high. Each neighbor $$B_i$$ also has high LRD. The ratios $$\approx 1$$ $$\Rightarrow$$ $$\text{LOF}_k(A) \approx 1$$.

**Case 2: $$A$$ is on the fringe of a dense cluster** 
Some neighbors are close (inside the cluster), but $$A$$ is farther away $$\Rightarrow$$ $$\text{lrd}_k(A)$$ is moderate. Neighbors inside the cluster have high LRD $$\Rightarrow$$ ratios $$> 1$$ $$\Rightarrow$$ $$\text{LOF}_k(A) > 1$$ but moderate.

**Case 3: $$A$$ is isolated, far from a dense cluster** 
Neighbors are far away $$\Rightarrow$$ $$\text{reach-dist}_k(A, B_i) = d(A, B_i)$$ (large) $$\Rightarrow$$ $$\text{lrd}_k(A)$$ is very low. Neighbors may be inside a dense cluster with high LRD $$\Rightarrow$$ ratios $$\gg 1$$ $$\Rightarrow$$ $$\text{LOF}_k(A) \gg 1$$.

### 5.6 Theoretical Properties

**Bounded LOF for points inside clusters**: For a point deep inside a cluster of uniform density $$\rho$$, all neighbors also have density $$\approx \rho$$, so:

$$
\text{LOF}_k(A) \approx \frac{\rho}{\rho} = 1
$$

**Sensitivity to $$k$$**: The choice of $$k$$ controls the locality scale:
* Small $$k$$: very local; sensitive to micro-clusters and noise
* Large $$k$$: more global; may miss small anomalous pockets
* Rule of thumb: try $$k \in [10, 50]$$ and take the maximum LOF across several $$k$$ values

### 5.7 Time and Space Complexity

| Phase | Complexity |
| --- | --- |
| $$k$$-NN computation | $$O(n^2 \cdot d)$$ naive, or $$O(n \log n \cdot d)$$ with KD-tree (low $$d$$) |
| LRD computation | $$O(n \cdot k)$$ |
| LOF computation | $$O(n \cdot k)$$ |
| **Total** | $$O(n^2 \cdot d)$$ without spatial indexing |

The quadratic cost makes LOF less scalable than Isolation Forest for very large datasets.

### 5.8 Strengths and Limitations

**Strengths**:
* Captures **local** density variations — handles datasets with clusters of different densities
* Produces a continuous outlier score with clear semantics
* No assumption about data distribution

**Limitations**:
* $$O(n^2)$$ complexity without indexing structures
* Sensitive to the choice of $$k$$
* Performance degrades in high dimensions (distance concentration)
* Not naturally incremental (recomputation needed for new points)

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor

np.random.seed(42)

# --- Create data with two clusters of DIFFERENT densities ---
# Dense cluster (sigma=0.3)
cluster_dense = np.random.randn(300, 2) * 0.3 + np.array([0, 0])
# Sparse cluster (sigma=1.5)
cluster_sparse = np.random.randn(200, 2) * 1.5 + np.array([7, 7])
# Anomalies near the dense cluster (would fool a global method)
anomalies = np.array([[2.5, 2.5], [2.0, 3.0], [-2.5, -2.5], [3.0, 1.5],
                      [10, 0], [-3, 5], [12, 12], [5, -3]])

X = np.vstack([cluster_dense, cluster_sparse, anomalies])
labels_true = np.array([1]*500 + [-1]*8)

# --- Fit LOF ---
lof = LocalOutlierFactor(
    n_neighbors=20,          # k
    contamination=0.05,
    metric='euclidean'
)
y_pred = lof.fit_predict(X)
lof_scores = -lof.negative_outlier_factor_  # higher = more anomalous

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter with LOF scores
scatter = axes[0].scatter(X[:, 0], X[:, 1], c=lof_scores, cmap='RdYlBu_r', s=20, alpha=0.7)
plt.colorbar(scatter, ax=axes[0], label='LOF Score')

# Mark detected anomalies
anom_mask = y_pred == -1
axes[0].scatter(X[anom_mask, 0], X[anom_mask, 1], facecolors='none',
                edgecolors='red', s=150, linewidths=2, label='Detected Anomaly')
axes[0].set_title('LOF Scores (color) with Detected Anomalies (red circles)', fontsize=12)
axes[0].legend()
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Score distribution
axes[1].hist(lof_scores[labels_true == 1], bins=40, alpha=0.7, color='steelblue', label='Normal', density=True)
axes[1].hist(lof_scores[labels_true == -1], bins=10, alpha=0.7, color='red', label='Anomaly', density=True)
axes[1].set_title('LOF Score Distribution', fontsize=13)
axes[1].set_xlabel('LOF Score (higher = more anomalous)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Detected anomalies: {(y_pred == -1).sum()} / {len(X)}")
print(f"True anomalies caught: {((y_pred == -1) & (labels_true == -1)).sum()} / {(labels_true == -1).sum()}")
print(f"\nTop 10 LOF scores:")
top_idx = np.argsort(lof_scores)[-10:][::-1]
for i in top_idx:
    label = 'ANOMALY' if labels_true[i] == -1 else 'normal'
    print(f"  Point {i}: LOF={lof_scores[i]:.3f}  [{label}]")

### 5.9 Industrial Example: Manufacturing Semiconductor Quality Control

**Scenario**: A semiconductor fabrication plant monitors wafer production. Each wafer has dozens of sensor measurements (temperature, pressure, gas flow, etch depth, film thickness). Defective wafers need to be caught before expensive downstream processing.

**Why LOF?**
* Different production "recipes" create clusters of different densities in sensor space
* A wafer that is normal for Recipe A might be anomalous for Recipe B
* LOF's local density comparison naturally handles this multi-modal structure
* The continuous LOF score enables prioritization (investigate highest LOF first)

**Key insight**: A global method like Isolation Forest might miss wafers that are subtly defective *within their own recipe cluster* because the sparse cluster of another recipe looks more anomalous globally. LOF's local comparison catches these local deviations.

In [0]:
import numpy as np
import pandas as pd
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Simulate semiconductor wafer sensor data ---
n_normal_A, n_normal_B, n_defective = 400, 300, 25

# Recipe A: tight tolerances (dense cluster)
recipe_A = pd.DataFrame({
    'temperature': np.random.normal(350, 2, n_normal_A),
    'pressure': np.random.normal(100, 1, n_normal_A),
    'gas_flow': np.random.normal(50, 0.5, n_normal_A),
    'etch_depth': np.random.normal(200, 3, n_normal_A),
    'film_thickness': np.random.normal(80, 1, n_normal_A),
    'recipe': 'A', 'defective': 0
})

# Recipe B: wider tolerances (sparse cluster)
recipe_B = pd.DataFrame({
    'temperature': np.random.normal(420, 8, n_normal_B),
    'pressure': np.random.normal(150, 5, n_normal_B),
    'gas_flow': np.random.normal(75, 3, n_normal_B),
    'etch_depth': np.random.normal(300, 10, n_normal_B),
    'film_thickness': np.random.normal(120, 4, n_normal_B),
    'recipe': 'B', 'defective': 0
})

# Defective wafers: subtle deviations within Recipe A
defective = pd.DataFrame({
    'temperature': np.random.normal(355, 3, n_defective),
    'pressure': np.random.normal(97, 2, n_defective),
    'gas_flow': np.random.normal(51.5, 1, n_defective),
    'etch_depth': np.random.normal(210, 5, n_defective),
    'film_thickness': np.random.normal(77, 2, n_defective),
    'recipe': 'A', 'defective': 1
})

df = pd.concat([recipe_A, recipe_B, defective], ignore_index=True).sample(frac=1, random_state=42)
features = ['temperature', 'pressure', 'gas_flow', 'etch_depth', 'film_thickness']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# --- Apply LOF ---
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
df['lof_pred'] = lof.fit_predict(X_scaled)
df['lof_score'] = -lof.negative_outlier_factor_

# --- Evaluation ---
y_true = df['defective']
y_pred = (df['lof_pred'] == -1).astype(int)

print("=== LOF: Semiconductor Wafer Quality Control ===")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Defective']))

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for recipe, marker, color in [('A', 'o', 'steelblue'), ('B', 's', 'seagreen')]:
    mask = (df['recipe'] == recipe) & (df['defective'] == 0)
    axes[0].scatter(df.loc[mask, 'temperature'], df.loc[mask, 'etch_depth'],
                    c=color, marker=marker, alpha=0.4, s=20, label=f'Recipe {recipe}')
defect_mask = df['defective'] == 1
axes[0].scatter(df.loc[defect_mask, 'temperature'], df.loc[defect_mask, 'etch_depth'],
                c='red', marker='x', s=80, label='Defective', zorder=5)
axes[0].set_title('Wafer Sensor Space', fontsize=12)
axes[0].legend()
axes[0].set_xlabel('Temperature')
axes[0].set_ylabel('Etch Depth')

for cat, mask, color in [
    ('Recipe A', (df['recipe']=='A') & (df['defective']==0), 'steelblue'),
    ('Recipe B', (df['recipe']=='B') & (df['defective']==0), 'seagreen'),
    ('Defective', df['defective']==1, 'red')
]:
    axes[1].hist(df.loc[mask, 'lof_score'], bins=30, alpha=0.6, color=color, label=cat, density=True)
axes[1].set_title('LOF Score by Category', fontsize=13)
axes[1].set_xlabel('LOF Score')
axes[1].legend()

plt.tight_layout()
plt.show()

## Chapter 6: One-Class Support Vector Machine (OC-SVM)

> *"The basic idea is to map the data into a feature space and find the hyperplane that separates the data from the origin with maximum margin."* 
> — Schölkopf et al. (2001)

### 6.1 Core Intuition

One-Class SVM (OC-SVM) is a **boundary-based** anomaly detection method. Unlike standard SVMs that separate two classes, OC-SVM learns a boundary around **a single class** (the normal data) and treats everything outside this boundary as anomalous.

**Geometric idea**: Map all training data (assumed to be normal) into a high-dimensional feature space via a kernel function $$\Phi$$. In this space, find a hyperplane that:
* Separates the data from the **origin** with **maximum margin**
* Allows a controlled fraction $$\nu$$ of training points to fall on the wrong side (soft margin)

The origin serves as a "default anomalous point" in the feature space. Normal data is mapped away from the origin; anomalies, being different, are expected to land near or beyond the origin-side of the hyperplane.

### 6.2 The $$\nu$$-Parameter

The parameter $$\nu \in (0, 1]$$ simultaneously controls:
* An **upper bound** on the fraction of training points classified as outliers
* A **lower bound** on the fraction of support vectors

This provides an elegant way to set the "contamination" rate without explicit thresholding.

### 6.3 Primal Formulation

Given training data $$\{x_1, \ldots, x_n\} \subset \mathbb{R}^d$$ and a feature map $$\Phi: \mathbb{R}^d \rightarrow \mathcal{H}$$ (where $$\mathcal{H}$$ is a reproducing kernel Hilbert space), the OC-SVM primal optimization problem is:

$$
\min_{w, \xi, \rho} \quad \frac{1}{2} \|w\|^2 + \frac{1}{\nu n} \sum_{i=1}^{n} \xi_i - \rho
$$

subject to:

$$
w \cdot \Phi(x_i) \geq \rho - \xi_i, \quad \xi_i \geq 0, \quad \forall\; i = 1, \ldots, n
$$

**Understanding each term**:
* $$\frac{1}{2}\|w\|^2$$: Regularization — encourages a smooth, generalizable boundary (maximum margin)
* $$\frac{1}{\nu n} \sum \xi_i$$: Penalty for violations — the slack variables $$\xi_i$$ allow some training points to fall on the wrong side
* $$-\rho$$: We maximize $$\rho$$ (the distance of the hyperplane from the origin) subject to most data lying on the positive side
* The constraint $$w \cdot \Phi(x_i) \geq \rho - \xi_i$$ says that each data point should be on the normal side ($$w \cdot \Phi(x_i) \geq \rho$$), with slack $$\xi_i$$ for violations

**Decision function**: For a new point $$x$$:

$$
f(x) = \text{sign}(w \cdot \Phi(x) - \rho)
$$

If $$f(x) = +1$$, the point is classified as normal. If $$f(x) = -1$$, it is an anomaly.

### 6.4 Dual Formulation — Detailed Derivation

#### Step 1: The Lagrangian

Introduce Lagrange multipliers $$\alpha_i \geq 0$$ for the main constraints and $$\beta_i \geq 0$$ for the non-negativity of slack:

$$
\mathcal{L}(w, \xi, \rho, \alpha, \beta) = \frac{1}{2}\|w\|^2 + \frac{1}{\nu n}\sum_{i=1}^n \xi_i - \rho - \sum_{i=1}^n \alpha_i\big(w \cdot \Phi(x_i) - \rho + \xi_i\big) - \sum_{i=1}^n \beta_i \xi_i
$$

#### Step 2: KKT Stationarity Conditions

Set the partial derivatives to zero:

**With respect to $$w$$:**

$$
\frac{\partial \mathcal{L}}{\partial w} = w - \sum_{i=1}^n \alpha_i \Phi(x_i) = 0 \quad \Rightarrow \quad \boxed{w = \sum_{i=1}^n \alpha_i \Phi(x_i)}
$$

This shows $$w$$ is a linear combination of the mapped training points (the representer theorem).

**With respect to $$\xi_i$$:**

$$
\frac{\partial \mathcal{L}}{\partial \xi_i} = \frac{1}{\nu n} - \alpha_i - \beta_i = 0 \quad \Rightarrow \quad \boxed{\alpha_i + \beta_i = \frac{1}{\nu n}}
$$

Since $$\beta_i \geq 0$$, this implies $$0 \leq \alpha_i \leq \frac{1}{\nu n}$$.

**With respect to $$\rho$$:**

$$
\frac{\partial \mathcal{L}}{\partial \rho} = -1 + \sum_{i=1}^n \alpha_i = 0 \quad \Rightarrow \quad \boxed{\sum_{i=1}^n \alpha_i = 1}
$$

#### Step 3: Substitution into the Lagrangian

Substitute the stationarity conditions back into $$\mathcal{L}$$:

$$
\|w\|^2 = w \cdot w = \sum_{i,j} \alpha_i \alpha_j \Phi(x_i) \cdot \Phi(x_j) = \sum_{i,j} \alpha_i \alpha_j K(x_i, x_j)
$$

where $$K(x_i, x_j) = \Phi(x_i) \cdot \Phi(x_j)$$ is the **kernel function**.

The $$\xi_i$$, $$\beta_i$$ terms cancel due to the stationarity condition, and the $$\rho$$ terms cancel because $$\sum \alpha_i = 1$$.

#### Step 4: The Dual Problem

The dual optimization problem becomes:

$$
\boxed{\min_{\alpha} \quad \frac{1}{2} \sum_{i=1}^n \sum_{j=1}^n \alpha_i \alpha_j K(x_i, x_j)}
$$

subject to:

$$
0 \leq \alpha_i \leq \frac{1}{\nu n}, \quad \sum_{i=1}^n \alpha_i = 1
$$

This is a **quadratic programming (QP)** problem in $$n$$ variables, solvable by standard QP solvers (e.g., SMO algorithm).

#### Step 5: Recovering $$\rho$$

From the KKT complementarity conditions, for any support vector $$x_j$$ with $$0 < \alpha_j < \frac{1}{\nu n}$$:

$$
\rho = w \cdot \Phi(x_j) = \sum_{i=1}^n \alpha_i K(x_i, x_j)
$$

#### Step 6: Decision Function (Kernel Form)

The final decision function, expressed entirely through the kernel (no explicit $$\Phi$$ needed):

$$
\boxed{f(x) = \text{sign}\left(\sum_{i=1}^n \alpha_i K(x_i, x) - \rho\right)}
$$

### 6.5 Common Kernels

| Kernel | $$K(x_i, x_j)$$ | Use Case |
| --- | --- | --- |
| Linear | $$x_i \cdot x_j$$ | Linearly separable data |
| Polynomial | $$(\gamma \cdot x_i \cdot x_j + r)^p$$ | Moderate non-linearity |
| **RBF (Gaussian)** | $$\exp(-\gamma \|x_i - x_j\|^2)$$ | **Most common for OC-SVM** |
| Sigmoid | $$\tanh(\gamma \cdot x_i \cdot x_j + r)$$ | Neural network-like |

The **RBF kernel** is preferred because it maps data to an infinite-dimensional space, providing maximum flexibility. The bandwidth parameter $$\gamma$$ controls the locality of the kernel:
* Large $$\gamma$$: very tight boundary (overfitting risk)
* Small $$\gamma$$: very loose boundary (underfitting risk)

### 6.6 Strengths and Limitations

**Strengths**:
* Theoretically well-founded with clear optimization objective
* The kernel trick enables non-linear boundaries without explicit feature computation
* $$\nu$$ parameter provides direct control over outlier fraction
* Effective in high dimensions (kernel methods are less affected by dimensionality)

**Limitations**:
* $$O(n^2)$$ to $$O(n^3)$$ training time (kernel matrix computation and QP solving)
* Sensitive to kernel choice and hyperparameters ($$\gamma$$, $$\nu$$)
* Does not scale well beyond $$\sim$$50K samples without approximations
* Semi-supervised: requires a (mostly) clean training set

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import OneClassSVM

np.random.seed(42)

# --- Generate synthetic data ---
# Normal data: crescent-shaped (non-linearly separable from origin)
theta = np.random.uniform(0, np.pi, 400)
r = 2 + np.random.randn(400) * 0.15
X_normal = np.column_stack([r * np.cos(theta), r * np.sin(theta)])

# Anomalies scattered
X_anomalies = np.random.uniform(low=-3, high=3, size=(15, 2))

X = np.vstack([X_normal, X_anomalies])
labels_true = np.array([1]*400 + [-1]*15)

# --- Fit One-Class SVM with RBF kernel ---
ocsvm = OneClassSVM(
    kernel='rbf',
    gamma=0.5,       # RBF bandwidth
    nu=0.05          # expected fraction of outliers
)
ocsvm.fit(X)

y_pred = ocsvm.predict(X)
scores = ocsvm.decision_function(X)

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-4, 4, 300), np.linspace(-2, 4, 300))
Z = ocsvm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

axes[0].contourf(xx, yy, Z, levels=np.linspace(Z.min(), Z.max(), 30), cmap='RdYlBu')
axes[0].contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)
normal_mask = y_pred == 1
axes[0].scatter(X[normal_mask, 0], X[normal_mask, 1], c='steelblue', s=15, alpha=0.6, label='Normal')
axes[0].scatter(X[~normal_mask, 0], X[~normal_mask, 1], c='red', s=60, marker='x', label='Anomaly')
axes[0].set_title('One-Class SVM (RBF) Decision Boundary', fontsize=13)
axes[0].legend()
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Score distribution
axes[1].hist(scores[labels_true == 1], bins=40, alpha=0.7, color='steelblue', label='Normal', density=True)
axes[1].hist(scores[labels_true == -1], bins=10, alpha=0.7, color='red', label='Anomaly', density=True)
axes[1].axvline(0, color='black', linestyle='--', label='Decision Boundary')
axes[1].set_title('Decision Function Distribution', fontsize=13)
axes[1].set_xlabel('Distance from Boundary')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Support vectors: {ocsvm.support_vectors_.shape[0]} / {len(X)}")
print(f"Detected anomalies: {(y_pred == -1).sum()} / {len(X)}")
print(f"True anomalies caught: {((y_pred == -1) & (labels_true == -1)).sum()} / {(labels_true == -1).sum()}")

### 6.7 Industrial Example: Network Intrusion Detection

**Scenario**: A corporate network security team monitors traffic features to detect intrusion attempts (port scans, DDoS, data exfiltration). The system is trained on a clean baseline of normal traffic, and must flag deviations in real time.

**Why One-Class SVM?**
* The security team has a clean baseline period ("known good" traffic) — perfect for semi-supervised OC-SVM
* Network traffic patterns are non-linear: certain combinations of features (packet size + frequency + port diversity) define attacks, not individual features alone
* The RBF kernel captures these non-linear patterns naturally
* $$\nu$$ can be set based on the expected false positive rate the SOC team can handle

**Feature engineering** for network intrusion: bytes per second, packets per second, unique destination ports, connection duration, protocol distribution, SYN/ACK ratio, and payload entropy.

In [0]:
import numpy as np
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Simulate network traffic data ---
n_normal, n_attack = 2000, 80

# Normal traffic
normal_traffic = pd.DataFrame({
    'bytes_per_sec': np.random.lognormal(mean=8, sigma=0.5, size=n_normal),
    'packets_per_sec': np.random.lognormal(mean=4, sigma=0.3, size=n_normal),
    'unique_dst_ports': np.random.poisson(lam=3, size=n_normal),
    'connection_duration': np.random.exponential(scale=30, size=n_normal),
    'syn_ack_ratio': np.random.beta(a=5, b=5, size=n_normal),
    'payload_entropy': np.random.normal(loc=4.5, scale=0.3, size=n_normal),
})
normal_traffic['is_attack'] = 0

# Attack traffic (port scan + DDoS characteristics)
attack_traffic = pd.DataFrame({
    'bytes_per_sec': np.random.lognormal(mean=10, sigma=1.0, size=n_attack),
    'packets_per_sec': np.random.lognormal(mean=7, sigma=0.8, size=n_attack),
    'unique_dst_ports': np.random.poisson(lam=50, size=n_attack),       # scanning many ports
    'connection_duration': np.random.exponential(scale=0.5, size=n_attack), # very short
    'syn_ack_ratio': np.random.beta(a=9, b=1, size=n_attack),           # mostly SYN (no ACK)
    'payload_entropy': np.random.normal(loc=7.5, scale=0.5, size=n_attack), # high entropy
})
attack_traffic['is_attack'] = 1

df = pd.concat([normal_traffic, attack_traffic], ignore_index=True).sample(frac=1, random_state=42)
features = ['bytes_per_sec', 'packets_per_sec', 'unique_dst_ports',
            'connection_duration', 'syn_ack_ratio', 'payload_entropy']

# --- Train on NORMAL traffic only (semi-supervised) ---
scaler = StandardScaler()
X_train_normal = scaler.fit_transform(normal_traffic[features])
X_all = scaler.transform(df[features])

ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
ocsvm.fit(X_train_normal)  # trained ONLY on normal data

# --- Predict on ALL data ---
df['ocsvm_pred'] = ocsvm.predict(X_all)
df['ocsvm_score'] = -ocsvm.decision_function(X_all)  # negate: higher = more anomalous

# --- Evaluation ---
y_true = df['is_attack']
y_pred = (df['ocsvm_pred'] == -1).astype(int)

print("=== One-Class SVM: Network Intrusion Detection ===")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Attack']))
print(f"ROC-AUC Score: {roc_auc_score(y_true, df['ocsvm_score']):.4f}")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'red', 'Attack')]:
    mask = df['is_attack'] == label
    axes[0].hist(df.loc[mask, 'ocsvm_score'], bins=50, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title('OC-SVM Anomaly Score Distribution', fontsize=13)
axes[0].set_xlabel('Anomaly Score')
axes[0].legend()

normal_mask = df['is_attack'] == 0
axes[1].scatter(df.loc[normal_mask, 'unique_dst_ports'], df.loc[normal_mask, 'packets_per_sec'],
                c='steelblue', alpha=0.3, s=10, label='Normal')
axes[1].scatter(df.loc[~normal_mask, 'unique_dst_ports'], df.loc[~normal_mask, 'packets_per_sec'],
                c='red', s=40, marker='x', label='Attack')
axes[1].set_title('Unique Dest Ports vs Packets/sec', fontsize=13)
axes[1].set_xlabel('Unique Destination Ports')
axes[1].set_ylabel('Packets per Second')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.show()

## Chapter 7: DBSCAN for Anomaly Detection

### 7.1 Core Intuition

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) by Ester et al. (1996) is primarily a clustering algorithm, but it has a built-in concept of **noise points** — points that do not belong to any cluster. These noise points are natural anomaly candidates.

### 7.2 Key Definitions

Given parameters $$\varepsilon$$ (neighborhood radius) and $$\text{MinPts}$$ (minimum number of points):

**$$\varepsilon$$-neighborhood**: The set of all points within distance $$\varepsilon$$ of point $$p$$:

$$
N_\varepsilon(p) = \{q \in \mathcal{D} \mid d(p, q) \leq \varepsilon\}
$$

**Core point**: A point $$p$$ is a core point if $$|N_\varepsilon(p)| \geq \text{MinPts}$$. It has enough neighbors to form a dense region.

**Border point**: A point $$p$$ is a border point if it is not a core point but lies within the $$\varepsilon$$-neighborhood of some core point.

**Noise point (anomaly)**: A point that is neither a core point nor a border point. It lies in a sparse region of the feature space.

### 7.3 The Algorithm

1. For each unvisited point $$p$$, compute $$N_\varepsilon(p)$$
2. If $$|N_\varepsilon(p)| \geq \text{MinPts}$$: start a new cluster and expand it by recursively adding all density-reachable points
3. If $$|N_\varepsilon(p)| < \text{MinPts}$$: label $$p$$ as noise (may later be claimed as a border point)

### 7.4 Anomaly Detection Interpretation

The noise points identified by DBSCAN are data points that:
* Live in regions too sparse to form clusters
* Are not reachable from any dense core point

This maps directly to the anomaly detection notion that anomalies reside in low-density regions.

### 7.5 Choosing $$\varepsilon$$ and MinPts

* **MinPts**: A common rule of thumb is $$\text{MinPts} \geq d + 1$$ where $$d$$ is the dimensionality, or simply $$\text{MinPts} = 2d$$. Higher values make the algorithm more conservative (fewer, denser clusters).
* **$$\varepsilon$$**: Use the **k-distance plot** — sort all $$k$$-nearest-neighbor distances in descending order and look for the "elbow." The $$\varepsilon$$ at the elbow separates dense regions from sparse noise.

### 7.6 Complexity

| Phase | Complexity |
| --- | --- |
| Without indexing | $$O(n^2 \cdot d)$$ |
| With KD-tree / Ball-tree | $$O(n \log n \cdot d)$$ for low $$d$$ |

### 7.7 Industrial Example: GPS Fleet Anomaly Detection

**Scenario**: A logistics company tracks GPS coordinates of its delivery fleet. Normal routes form dense clusters along highways and delivery zones. Vehicles deviating from expected routes (unauthorized detours, vehicle theft, driver behavior anomalies) appear as noise points.

**Why DBSCAN?**
* GPS routes form natural spatial clusters of arbitrary shape (unlike k-Means)
* No need to specify the number of clusters in advance
* Noise label directly identifies off-route behavior
* Works well in 2D/3D spatial data where distance is meaningful

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# --- Simulate GPS fleet data ---
# Route 1: Highway corridor
t1 = np.linspace(0, 1, 300)
route1 = np.column_stack([t1 * 50 + np.random.normal(0, 0.3, 300),
                          t1 * 20 + np.random.normal(0, 0.3, 300)])

# Route 2: Urban delivery zone
theta = np.random.uniform(0, 2 * np.pi, 250)
r = np.random.normal(3, 0.5, 250)
route2 = np.column_stack([30 + r * np.cos(theta), 25 + r * np.sin(theta)])

# Route 3: Suburban delivery cluster
route3 = np.random.normal(loc=[10, 30], scale=[1.5, 1.5], size=(200, 2))

# Anomalous GPS points (detours, unauthorized locations)
anomalies = np.array([[5, 5], [45, 35], [60, 5], [25, 40], [0, 20],
                      [55, 30], [15, 10], [40, 0], [35, 38], [50, 10]])

X = np.vstack([route1, route2, route3, anomalies])
labels_true = np.array([0]*300 + [0]*250 + [0]*200 + [-1]*10)  # 0=normal, -1=anomaly

# --- Apply DBSCAN ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

dbscan = DBSCAN(
    eps=0.3,          # neighborhood radius
    min_samples=10,   # MinPts
    metric='euclidean'
)
labels = dbscan.fit_predict(X_scaled)

# --- Analyze results ---
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_mask = labels == -1

print("=== DBSCAN: Fleet GPS Anomaly Detection ===")
print(f"Clusters found: {n_clusters}")
print(f"Noise points (anomalies): {noise_mask.sum()} / {len(X)}")
print(f"True anomalies caught: {((noise_mask) & (labels_true == -1)).sum()} / {(labels_true == -1).sum()}")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Original data with true labels
axes[0].scatter(X[labels_true == 0, 0], X[labels_true == 0, 1],
                c='steelblue', s=10, alpha=0.5, label='Normal routes')
axes[0].scatter(anomalies[:, 0], anomalies[:, 1],
                c='red', s=80, marker='x', label='True anomalies')
axes[0].set_title('Ground Truth', fontsize=13)
axes[0].legend()
axes[0].set_xlabel('Longitude (scaled)')
axes[0].set_ylabel('Latitude (scaled)')

# DBSCAN results
unique_labels = set(labels)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
for k, col in zip(sorted(unique_labels), colors):
    mask = labels == k
    if k == -1:
        axes[1].scatter(X[mask, 0], X[mask, 1], c='red', s=80, marker='x', label='Noise (anomaly)')
    else:
        axes[1].scatter(X[mask, 0], X[mask, 1], c=[col], s=10, alpha=0.5, label=f'Cluster {k}')
axes[1].set_title('DBSCAN Clustering Result', fontsize=13)
axes[1].legend()
axes[1].set_xlabel('Longitude (scaled)')
axes[1].set_ylabel('Latitude (scaled)')

plt.tight_layout()
plt.show()

## Chapter 8: Elliptic Envelope (Mahalanobis Distance)

### 8.1 Core Intuition

The Elliptic Envelope method assumes the normal data is generated from a **multivariate Gaussian distribution** and identifies anomalies as points that fall far from the center of this distribution, measured by the **Mahalanobis distance**.

### 8.2 Mahalanobis Distance

For a point $$x \in \mathbb{R}^d$$ with data having mean $$\mu$$ and covariance matrix $$\Sigma$$, the Mahalanobis distance is:

$$
D_M(x) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}
$$

**Why not Euclidean distance?** Euclidean distance treats all dimensions equally, ignoring correlations. The Mahalanobis distance:
* Accounts for **correlations** between features
* **Scales** each dimension by its variance
* Measures distance in units of standard deviation along each principal axis

**Geometric interpretation**: The Mahalanobis distance defines **ellipsoids** in feature space (iso-distance contours). Points with $$D_M(x) > \tau$$ lie outside the ellipsoid and are flagged as anomalous.

### 8.3 The Chi-Squared Connection

If $$x \sim \mathcal{N}(\mu, \Sigma)$$, then $$D_M^2(x)$$ follows a **chi-squared distribution** with $$d$$ degrees of freedom:

$$
D_M^2(x) \sim \chi^2_d
$$

This provides a principled threshold: flag as anomalous if $$D_M^2(x) > \chi^2_{d, 1-\alpha}$$ where $$\alpha$$ is the desired significance level. For example, with $$d = 5$$ features and $$\alpha = 0.01$$: $$\chi^2_{5, 0.99} \approx 15.09$$.

### 8.4 Robust Covariance Estimation (MCD)

The sample mean and covariance are sensitive to outliers (the same masking effect as in Z-scores). The Elliptic Envelope uses the **Minimum Covariance Determinant (MCD)** estimator:

$$
(\hat{\mu}_{\text{MCD}}, \hat{\Sigma}_{\text{MCD}}) = \arg\min_{\mu, \Sigma} \det(\Sigma)
$$

subject to the constraint that the estimates are computed from the subset of $$h$$ points (out of $$n$$) that minimizes the determinant. Typically $$h = \lfloor (n + d + 1) / 2 \rfloor$$.

The MCD estimator is **robust** — it can tolerate up to $$(n - h)/n$$ fraction of outliers without breaking down.

### 8.5 Algorithm Summary

1. Estimate $$\hat{\mu}$$ and $$\hat{\Sigma}$$ using robust MCD
2. Compute $$D_M^2(x_i)$$ for each data point
3. Flag points where $$D_M^2(x_i) > \chi^2_{d, 1-\alpha}$$ as anomalies

### 8.6 Strengths and Limitations

**Strengths**:
* Statistically principled with a clear probability model
* Accounts for feature correlations (unlike Z-score per dimension)
* MCD estimator makes it robust to contaminated training data
* Produces a meaningful distance metric (Mahalanobis distance in standard deviation units)

**Limitations**:
* Assumes unimodal Gaussian data — fails with multi-modal or non-elliptical distributions
* MCD becomes unreliable when $$d > n$$ (more features than samples)
* Not suitable for categorical or mixed-type data

### 8.7 Industrial Example: Pharmaceutical Process Monitoring

**Scenario**: A pharmaceutical company monitors batch chemical reactions. Each batch has correlated measurements (temperature profile, pH trajectory, viscosity, yield). Deviations indicate a batch failure that could produce unsafe medication.

**Why Elliptic Envelope?** Process variables are continuous, correlated, and approximately Gaussian under normal operating conditions. The Mahalanobis distance captures multivariate deviations that univariate methods would miss.

In [0]:
import numpy as np
import pandas as pd
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Simulate pharmaceutical batch process data ---
n_good, n_bad = 500, 20

# Correlated normal process (temperature, pH, viscosity, yield are correlated)
mean_normal = [150, 7.0, 45, 92]
cov_normal = [
    [25,  2,  5, -3],     # temperature variance=25, correlated with others
    [ 2,  0.04, 0.3, 0.1],
    [ 5,  0.3,  9,  -2],
    [-3,  0.1, -2,   4]
]
good_batches = np.random.multivariate_normal(mean_normal, cov_normal, n_good)

# Failed batches: multivariate deviation (not just one feature off)
bad_mean = [158, 6.5, 52, 85]
bad_cov = [
    [30, -1, 8, 2],
    [-1, 0.09, -0.2, -0.1],
    [8, -0.2, 16, -3],
    [2, -0.1, -3, 9]
]
bad_batches = np.random.multivariate_normal(bad_mean, bad_cov, n_bad)

X = np.vstack([good_batches, bad_batches])
labels_true = np.array([0]*n_good + [1]*n_bad)  # 0=normal, 1=defective

df = pd.DataFrame(X, columns=['temperature', 'pH', 'viscosity', 'yield_pct'])
df['defective'] = labels_true

# --- Apply Elliptic Envelope ---
ee = EllipticEnvelope(
    contamination=0.05,    # expected outlier fraction
    support_fraction=0.8,  # MCD subset fraction
    random_state=42
)
df['ee_pred'] = ee.fit_predict(X)  # +1 normal, -1 anomaly
df['mahal_dist'] = ee.mahalanobis(X)

# --- Evaluation ---
y_true = df['defective']
y_pred = (df['ee_pred'] == -1).astype(int)

print("=== Elliptic Envelope: Pharmaceutical Process Monitoring ===")
print(classification_report(y_true, y_pred, target_names=['Good Batch', 'Failed Batch']))

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: temperature vs yield (correlated features)
good_mask = df['defective'] == 0
axes[0].scatter(df.loc[good_mask, 'temperature'], df.loc[good_mask, 'yield_pct'],
                c='steelblue', alpha=0.5, s=20, label='Good batch')
axes[0].scatter(df.loc[~good_mask, 'temperature'], df.loc[~good_mask, 'yield_pct'],
                c='red', s=60, marker='x', label='Failed batch')

# Draw ellipse (approximate from Mahalanobis)
from matplotlib.patches import Ellipse
mean_xy = ee.location_[[0, 3]]
cov_xy = ee.covariance_[np.ix_([0, 3], [0, 3])]
eig_vals, eig_vecs = np.linalg.eigh(cov_xy)
angle = np.degrees(np.arctan2(eig_vecs[1, 1], eig_vecs[0, 1]))
for n_std, alpha in [(2, 0.3), (3, 0.15)]:
    width, height = 2 * n_std * np.sqrt(eig_vals)
    ell = Ellipse(xy=mean_xy, width=width, height=height, angle=angle,
                  fill=False, edgecolor='darkred', linestyle='--', linewidth=1.5, alpha=alpha)
    axes[0].add_patch(ell)

axes[0].set_title('Temperature vs Yield with Mahalanobis Ellipses', fontsize=12)
axes[0].legend()
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Yield (%)')

# Mahalanobis distance distribution
axes[1].hist(df.loc[good_mask, 'mahal_dist'], bins=30, alpha=0.7, color='steelblue', label='Good', density=True)
axes[1].hist(df.loc[~good_mask, 'mahal_dist'], bins=10, alpha=0.7, color='red', label='Failed', density=True)
from scipy.stats import chi2
chi2_threshold = chi2.ppf(0.975, df=4)
axes[1].axvline(chi2_threshold, color='black', linestyle='--', label=f'$\\chi^2$ threshold ({chi2_threshold:.1f})')
axes[1].set_title('Mahalanobis Distance Distribution', fontsize=13)
axes[1].set_xlabel('Squared Mahalanobis Distance')
axes[1].legend()

plt.tight_layout()
plt.show()

## Chapter 9: Autoencoders for Anomaly Detection

### 9.1 Core Intuition

An **autoencoder** is a neural network trained to reconstruct its input. It compresses the input through a **bottleneck** (encoder) and then tries to rebuild it (decoder). The key insight for anomaly detection:

* Train the autoencoder on **normal data only**
* The network learns to reconstruct normal patterns well
* When an anomalous input is presented, the network **fails to reconstruct it accurately**
* The **reconstruction error** serves as the anomaly score

### 9.2 Architecture

$$
x \xrightarrow{\text{Encoder}} z \xrightarrow{\text{Decoder}} \hat{x}
$$

**Encoder** $$f_\theta$$: Maps input $$x \in \mathbb{R}^d$$ to a latent representation $$z \in \mathbb{R}^k$$ where $$k \ll d$$:

$$
z = f_\theta(x) = \sigma(W_e x + b_e)
$$

**Decoder** $$g_\phi$$: Reconstructs the input from the latent representation:

$$
\hat{x} = g_\phi(z) = \sigma(W_d z + b_d)
$$

where $$\sigma$$ is an activation function (ReLU, sigmoid, etc.).

### 9.3 Loss Function (Optimization Objective)

The autoencoder minimizes the **mean squared reconstruction error** over the training set:

$$
\mathcal{L}(\theta, \phi) = \frac{1}{n} \sum_{i=1}^n \|x_i - \hat{x}_i\|^2 = \frac{1}{n} \sum_{i=1}^n \|x_i - g_\phi(f_\theta(x_i))\|^2
$$

This is optimized via **backpropagation** and gradient descent (Adam, SGD, etc.).

### 9.4 Anomaly Score

For a new instance $$x$$, the anomaly score is the reconstruction error:

$$
s(x) = \|x - g_\phi(f_\theta(x))\|^2 = \|x - \hat{x}\|^2
$$

* **Normal points**: low reconstruction error (the autoencoder has learned their patterns)
* **Anomalous points**: high reconstruction error (they don't match learned patterns)

### 9.5 Why the Bottleneck Matters

The bottleneck ($$k \ll d$$) forces the network to learn a **compressed representation** of the normal data manifold. If the latent dimension equals the input dimension ($$k = d$$), the network could simply learn the identity function, making it useless for anomaly detection.

The bottleneck dimension $$k$$ controls the trade-off:
* Too small: even normal data has high reconstruction error
* Too large: anomalies can also be reconstructed well
* Optimal $$k$$ depends on the intrinsic dimensionality of the normal data

### 9.6 Variants

| Variant | Key Difference | Advantage |
| --- | --- | --- |
| **Vanilla AE** | Simple encoder-decoder | Baseline; fast to train |
| **Denoising AE** | Input corrupted with noise; must reconstruct clean version | More robust features |
| **Variational AE (VAE)** | Latent space has probabilistic structure ($$z \sim \mathcal{N}(\mu, \sigma)$$) | Smoother latent space; probabilistic anomaly score |
| **Sparse AE** | Regularization forces most latent units to be inactive | Learns selective features |
| **LSTM AE** | Uses LSTM layers for sequential data | Time-series anomaly detection |

### 9.7 Strengths and Limitations

**Strengths**:
* Handles high-dimensional, complex data (images, time series, text embeddings)
* Learns non-linear manifolds (unlike PCA)
* Flexible: architecture can be adapted to any data modality
* Can detect subtle anomalies that linear methods miss

**Limitations**:
* Requires sufficient normal training data
* Hyperparameter-heavy (architecture, learning rate, epochs, latent dimension)
* Training can be unstable; risk of overfitting to training noise
* Less interpretable than statistical or tree-based methods
* Anomalous patterns seen during training will have low reconstruction error (data leakage)

### 9.8 Industrial Example: IoT Sensor Anomaly Detection

**Scenario**: A smart building system monitors hundreds of IoT sensors (temperature, humidity, CO2, light, occupancy). Equipment failures, HVAC malfunctions, or sensor drift cause multivariate anomalies in the sensor stream.

**Why Autoencoders?**
* Hundreds of correlated sensor features — traditional methods struggle in this dimensionality
* The autoencoder learns the normal inter-sensor correlation structure
* When a sensor drifts or equipment fails, reconstruction error spikes for the affected sensors
* The per-feature reconstruction error can pinpoint *which* sensors are anomalous (interpretability boost)

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# --- Use a simple autoencoder with sklearn's MLPRegressor as lightweight proxy ---
# (For production, use PyTorch/TensorFlow; this keeps dependencies minimal)
from sklearn.neural_network import MLPRegressor

np.random.seed(42)

# --- Simulate IoT sensor data (12 sensors) ---
n_normal_train = 3000  # training: normal only
n_normal_test  = 1000
n_anomaly_test = 50

# Normal data: correlated sensors
base = np.random.randn(n_normal_train + n_normal_test, 3)  # 3 latent factors
W = np.random.randn(3, 12) * 0.5  # factor loadings to 12 sensors
noise = np.random.randn(n_normal_train + n_normal_test, 12) * 0.2
normal_all = base @ W + noise

X_train = normal_all[:n_normal_train]
X_test_normal = normal_all[n_normal_train:]

# Anomalous: sensor drift and equipment failure patterns
base_anom = np.random.randn(n_anomaly_test, 3)
X_test_anomaly = base_anom @ W + np.random.randn(n_anomaly_test, 12) * 0.2
# Inject anomaly: sensors 0-2 drift significantly
X_test_anomaly[:, 0] += np.random.uniform(3, 6, n_anomaly_test)
X_test_anomaly[:, 1] += np.random.uniform(-5, -3, n_anomaly_test)
X_test_anomaly[:, 2] *= 3  # amplified readings

X_test = np.vstack([X_test_normal, X_test_anomaly])
y_test = np.array([0]*n_normal_test + [1]*n_anomaly_test)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Build Autoencoder (encoder: 12->8->4, decoder: 4->8->12) ---
# Using MLPRegressor to reconstruct input from itself = autoencoder behavior
autoencoder = MLPRegressor(
    hidden_layer_sizes=(8, 4, 8),  # bottleneck at 4
    activation='relu',
    solver='adam',
    max_iter=200,
    learning_rate_init=0.001,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

# Train: input = output = normal data
autoencoder.fit(X_train_scaled, X_train_scaled)

# --- Compute reconstruction error ---
X_reconstructed = autoencoder.predict(X_test_scaled)
recon_error = np.mean((X_test_scaled - X_reconstructed) ** 2, axis=1)

# Threshold: 95th percentile of training reconstruction error
X_train_recon = autoencoder.predict(X_train_scaled)
train_error = np.mean((X_train_scaled - X_train_recon) ** 2, axis=1)
threshold = np.percentile(train_error, 95)

y_pred = (recon_error > threshold).astype(int)

print("=== Autoencoder: IoT Sensor Anomaly Detection ===")
print(f"Reconstruction error threshold (95th pctl): {threshold:.4f}")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomaly']))

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Reconstruction error distribution
axes[0].hist(recon_error[y_test == 0], bins=50, alpha=0.7, color='steelblue', label='Normal', density=True)
axes[0].hist(recon_error[y_test == 1], bins=20, alpha=0.7, color='red', label='Anomaly', density=True)
axes[0].axvline(threshold, color='black', linestyle='--', label=f'Threshold={threshold:.3f}')
axes[0].set_title('Reconstruction Error Distribution', fontsize=13)
axes[0].set_xlabel('Mean Squared Error')
axes[0].legend()

# Per-sensor reconstruction error for anomalous points
sensor_errors_anom = np.mean((X_test_scaled[y_test == 1] - X_reconstructed[y_test == 1]) ** 2, axis=0)
sensor_errors_norm = np.mean((X_test_scaled[y_test == 0] - X_reconstructed[y_test == 0]) ** 2, axis=0)

x_pos = np.arange(12)
axes[1].bar(x_pos - 0.2, sensor_errors_norm, 0.4, color='steelblue', alpha=0.7, label='Normal')
axes[1].bar(x_pos + 0.2, sensor_errors_anom, 0.4, color='red', alpha=0.7, label='Anomaly')
axes[1].set_title('Per-Sensor Reconstruction Error', fontsize=13)
axes[1].set_xlabel('Sensor Index')
axes[1].set_ylabel('MSE')
axes[1].set_xticks(x_pos)
axes[1].legend()

# Training loss curve
axes[2].plot(autoencoder.loss_curve_, color='steelblue', linewidth=2)
axes[2].set_title('Autoencoder Training Loss', fontsize=13)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss (MSE)')

plt.tight_layout()
plt.show()

## Chapter 10: Comparative Analysis and Practical Guidelines

### 10.1 Algorithm Comparison Matrix

| Criterion | Isolation Forest | LOF | One-Class SVM | DBSCAN | Elliptic Envelope | Autoencoder |
| --- | --- | --- | --- | --- | --- | --- |
| **Paradigm** | Unsupervised | Unsupervised | Semi-supervised | Unsupervised | Semi-supervised | Semi-supervised |
| **Core Idea** | Isolation via random splits | Local density ratio | Max-margin boundary | Density clustering | Mahalanobis distance | Reconstruction error |
| **Training Time** | $$O(t \cdot \psi \log \psi)$$ | $$O(n^2 d)$$ | $$O(n^2)$$ to $$O(n^3)$$ | $$O(n \log n)$$ with index | $$O(n d^2)$$ | $$O(\text{epochs} \cdot n \cdot \text{params})$$ |
| **Scalability** | Excellent | Poor | Poor | Good | Moderate | Good (with GPU) |
| **Handles Local Anomalies** | Weak | Strong | Moderate | Strong | Weak | Moderate |
| **High Dimensionality** | Good | Degrades | Good (with kernel) | Degrades | Poor if $$d > n$$ | Excellent |
| **Interpretability** | Moderate (path length) | Good (LOF score) | Low | Good (noise label) | Good (distance) | Low |
| **Key Hyperparameters** | $$t$$, $$\psi$$, contamination | $$k$$, contamination | $$\nu$$, kernel, $$\gamma$$ | $$\varepsilon$$, MinPts | contamination | Architecture, $$k$$, epochs |
| **Assumes Distribution** | No | No | No | No | Yes (Gaussian) | No |

### 10.2 Decision Framework: Which Algorithm to Choose?

**Start with Isolation Forest** when:
* You have a large dataset ($$>$$100K samples)
* You want a fast, general-purpose baseline
* Global anomalies are the primary concern

**Use LOF** when:
* Your data has clusters of different densities
* Local anomalies matter (e.g., anomalies near but not inside a specific cluster)
* Dataset is moderate-sized ($$<$$50K)

**Use One-Class SVM** when:
* You have clean training data (semi-supervised setting)
* The data has complex, non-linear boundaries
* Dataset is small to moderate ($$<$$20K)

**Use DBSCAN** when:
* Your data is spatial or has natural cluster structure
* You also want the clustering itself, not just anomaly labels
* The number of clusters is unknown

**Use Elliptic Envelope** when:
* Features are continuous and approximately Gaussian
* Feature correlations are important
* You need a statistically principled, interpretable method

**Use Autoencoders** when:
* Data is high-dimensional (images, time series, embeddings)
* Non-linear structure is complex and low-dimensional
* You have sufficient normal training data

### 10.3 Ensemble Approaches

In practice, **combining multiple anomaly detectors** often outperforms any single method:

* **Score averaging**: Normalize scores from each detector to $$[0, 1]$$ and average
* **Majority voting**: Flag a point as anomalous if $$\geq k$$ out of $$m$$ detectors agree
* **Maximum score**: Flag based on the maximum anomaly score across detectors (catches anomalies that any detector finds)

PyOD (Python Outlier Detection) library provides a unified API for ensembling.

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.covariance import EllipticEnvelope
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score
import time

np.random.seed(42)

# --- Generate benchmark dataset ---
n_normal, n_anom = 2000, 50
cluster1 = np.random.randn(n_normal // 2, 2) * 0.6 + [2, 2]
cluster2 = np.random.randn(n_normal // 2, 2) * 1.2 + [-3, -3]
anomalies = np.random.uniform(-6, 6, (n_anom, 2))

X = np.vstack([cluster1, cluster2, anomalies])
y_true = np.array([0]*n_normal + [1]*n_anom)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Run all detectors ---
results = {}

# 1. Isolation Forest
t0 = time.time()
iforest = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)
iforest.fit(X_scaled)
scores_if = -iforest.score_samples(X_scaled)
results['Isolation Forest'] = {'scores': scores_if, 'time': time.time() - t0}

# 2. LOF
t0 = time.time()
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.03)
lof.fit_predict(X_scaled)
scores_lof = -lof.negative_outlier_factor_
results['LOF'] = {'scores': scores_lof, 'time': time.time() - t0}

# 3. One-Class SVM
t0 = time.time()
ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.03)
ocsvm.fit(X_scaled)
scores_ocsvm = -ocsvm.decision_function(X_scaled)
results['One-Class SVM'] = {'scores': scores_ocsvm, 'time': time.time() - t0}

# 4. Elliptic Envelope
t0 = time.time()
ee = EllipticEnvelope(contamination=0.03, random_state=42)
ee.fit(X_scaled)
scores_ee = -ee.decision_function(X_scaled)
results['Elliptic Envelope'] = {'scores': scores_ee, 'time': time.time() - t0}

# 5. DBSCAN (score = binary: noise or not)
t0 = time.time()
dbscan = DBSCAN(eps=0.5, min_samples=10)
labels_db = dbscan.fit_predict(X_scaled)
scores_db = (labels_db == -1).astype(float)
results['DBSCAN'] = {'scores': scores_db, 'time': time.time() - t0}

# 6. Autoencoder
t0 = time.time()
ae = MLPRegressor(hidden_layer_sizes=(8, 3, 8), activation='relu', max_iter=200,
                  learning_rate_init=0.001, random_state=42, early_stopping=True)
ae.fit(X_scaled[:n_normal], X_scaled[:n_normal])  # train on normal only
X_recon = ae.predict(X_scaled)
scores_ae = np.mean((X_scaled - X_recon) ** 2, axis=1)
results['Autoencoder'] = {'scores': scores_ae, 'time': time.time() - t0}

# --- Compute metrics ---
print("=== Comparative Benchmark ===")
print(f"{'Method':<20} {'ROC-AUC':>10} {'Time (s)':>10}")
print("-" * 42)
for name, r in results.items():
    try:
        auc = roc_auc_score(y_true, r['scores'])
    except ValueError:
        auc = float('nan')
    print(f"{name:<20} {auc:>10.4f} {r['time']:>10.4f}")

# --- Visualization: Score distributions side by side ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (name, r) in zip(axes.ravel(), results.items()):
    ax.hist(r['scores'][y_true == 0], bins=40, alpha=0.7, color='steelblue', label='Normal', density=True)
    ax.hist(r['scores'][y_true == 1], bins=15, alpha=0.7, color='red', label='Anomaly', density=True)
    try:
        auc = roc_auc_score(y_true, r['scores'])
        ax.set_title(f"{name} (AUC={auc:.3f})", fontsize=12)
    except ValueError:
        ax.set_title(f"{name}", fontsize=12)
    ax.set_xlabel('Anomaly Score')
    ax.legend(fontsize=9)

plt.suptitle('Anomaly Score Distributions Across Methods', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## References and Further Reading

### Foundational Papers

* **Isolation Forest**: Liu, F.T., Ting, K.M., and Zhou, Z.H. (2008). *Isolation Forest*. IEEE International Conference on Data Mining (ICDM). The original paper introducing the isolation-based approach with the BST normalization derivation.

* **Local Outlier Factor**: Breunig, M.M., Kriegel, H.P., Ng, R.T., and Sander, J. (2000). *LOF: Identifying Density-Based Local Outliers*. ACM SIGMOD International Conference on Management of Data.

* **One-Class SVM**: Schölkopf, B., Platt, J.C., Shawe-Taylor, J., Smola, A.J., and Williamson, R.C. (2001). *Estimating the Support of a High-Dimensional Distribution*. Neural Computation, 13(7).

* **DBSCAN**: Ester, M., Kriegel, H.P., Sander, J., and Xu, X. (1996). *A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise*. KDD.

* **Minimum Covariance Determinant**: Rousseeuw, P.J. (1984). *Least Median of Squares Regression*. Journal of the American Statistical Association.

* **Autoencoders for Anomaly Detection**: Sakurada, M. and Yairi, T. (2014). *Anomaly Detection Using Autoencoders with Nonlinear Dimensionality Reduction*. MLSDA Workshop.

### Recommended Books

* Aggarwal, C.C. (2017). *Outlier Analysis*. Springer. Comprehensive textbook covering all major families of anomaly detection methods.
* Chandola, V., Banerjee, A., and Kumar, V. (2009). *Anomaly Detection: A Survey*. ACM Computing Surveys. Widely cited survey covering taxonomy, methods, and applications.

### Software Libraries

* **scikit-learn**: `sklearn.ensemble.IsolationForest`, `sklearn.neighbors.LocalOutlierFactor`, `sklearn.svm.OneClassSVM`, `sklearn.covariance.EllipticEnvelope`
* **PyOD**: Python Outlier Detection library with 40+ algorithms and ensemble methods (`pip install pyod`)
* **Alibi Detect**: Outlier, adversarial, and drift detection (`pip install alibi-detect`)

---

*Notebook authored as a comprehensive reference guide to anomaly detection algorithms. All code examples use synthetic data to illustrate core concepts and are self-contained.*